In [1]:
# @title Complete 3-File Translation & Glossary Package Generator
# @markdown Select your translation direction, enter your source text, and run the cell.

TRANSLATION_DIRECTION = "English to Ukrainian" # @param ['English to Ukrainian', 'Ukrainian to English']
SOURCE_TEXT = """Consecutive Translation Trainer (Pro) is an advanced, flexible software suite engineered specifically for interpreters, translation students, and language professionals. By bridging state-of-the-art automatic speech recognition, natural language segmentation, and localized machine translation pipelines, this tool provides a comprehensive environment to practice, transcribe, segment, translate, and archive consecutive interpretation sessions. Terminology management ensures consistency across large translation projects and technical documents.""" # @param {type:"string"}
MAX_GLOSSARY_TERMS = 100 # @param {type:"integer"}
ZIP_FILENAME = "translation_package.zip"

# 1. Install required libraries
print("Installing required packages (python-docx, spacy, deep-translator)...")
!pip install -q python-docx spacy deep-translator
!python -m spacy download en_core_web_sm
!python -m spacy download uk_core_news_sm 2>/dev/null || true

import os
import zipfile
import docx
import spacy
from deep_translator import GoogleTranslator
from collections import Counter
from google.colab import files

print("Setup completed successfully!")

# 2. Configure languages and SpaCy model based on direction
if TRANSLATION_DIRECTION == "English to Ukrainian":
    src_lang, tgt_lang = 'en', 'uk'
    nlp = spacy.load("en_core_web_sm")
    src_title, tgt_title = "English Term", "Ukrainian Term"
else:
    src_lang, tgt_lang = 'uk', 'en'
    try:
        nlp = spacy.load("uk_core_news_sm")
    except Exception:
        nlp = spacy.load("en_core_web_sm")
    src_title, tgt_title = "Ukrainian Term", "English Term"

translator = GoogleTranslator(source=src_lang, target=tgt_lang)

def translate_paragraph(text):
    """Safely translates paragraph text, handling character limits and chunking if needed."""
    if not text or not text.strip():
        return ""
    if len(text) > 4000:
        chunks = [text[i:i+4000] for i in range(0, len(text), 4000)]
        translated_chunks = [translator.translate(chunk) for chunk in chunks]
        return " ".join([tc for tc in translated_chunks if tc])
    try:
        result = translator.translate(text)
        return result if result else text
    except Exception as e:
        return f"[Translation Error: {str(e)}]"

def extract_and_translate_glossary(text, max_terms=100):
    """Extracts noun chunks and key terms using SpaCy, translating each individually."""
    doc = nlp(text)
    candidates = []

    # Extract noun chunks first
    for chunk in doc.noun_chunks:
        term = chunk.text.strip()
        if len(term) > 2 and term.lower() not in [c.lower() for c in candidates]:
            candidates.append(term)

    # Supplement with important nouns, proper nouns, and adjectives if needed
    if len(candidates) < max_terms:
        for token in doc:
            if token.pos_ in ["NOUN", "PROPN", "ADJ"] and not token.is_stop and len(token.text) > 2:
                term = token.text.strip()
                if term.lower() not in [c.lower() for c in candidates]:
                    candidates.append(term)

    # Sort by frequency
    counts = Counter([c.lower() for c in candidates])
    top_terms = [t for t, _ in counts.most_common(max_terms * 2)]

    glossary = []
    seen = set()
    for term in top_terms:
        if term in seen:
            continue
        seen.add(term)

        # Translate each term individually
        try:
            translated_term = translator.translate(term)
        except Exception:
            translated_term = term

        if translated_term and translated_term.strip():
            glossary.append((term.capitalize(), translated_term.capitalize()))

        if len(glossary) >= max_terms:
            break

    return glossary

def generate_package():
    print("Step 1/4: Splitting text and executing true neural translation...")
    paragraphs = [p.strip() for p in SOURCE_TEXT.split('\n') if p.strip()]
    if not paragraphs:
        paragraphs = [SOURCE_TEXT]

    translated_paragraphs = []
    for p in paragraphs:
        translated_paragraphs.append(translate_paragraph(p))

    print("Step 2/4: Extracting and translating glossary terms...")
    glossary_items = extract_and_translate_glossary(SOURCE_TEXT, max_terms=MAX_GLOSSARY_TERMS)

    print("Step 3/4: Compiling three separate Word documents (.docx)...")

    # 1. Original Document
    doc_orig = docx.Document()
    doc_orig.add_heading("Original Document", level=1)
    for p in paragraphs:
        doc_orig.add_paragraph(p)
    file_orig = "1_Original_Document.docx"
    doc_orig.save(file_orig)

    # 2. Translated Document
    doc_trans = docx.Document()
    doc_trans.add_heading("Translated Document", level=1)
    for p in translated_paragraphs:
        doc_trans.add_paragraph(p if p else "[No translation]")
    file_trans = "2_Translated_Document.docx"
    doc_trans.save(file_trans)

    # 3. Glossary Document
    doc_gloss = docx.Document()
    doc_gloss.add_heading(f"Glossary of Terms ({len(glossary_items)} Key Terms)", level=1)

    if glossary_items:
        table = doc_gloss.add_table(rows=1, cols=2)
        table.style = 'Table Grid'
        hdr_cells = table.rows[0].cells
        hdr_cells[0].text = src_title
        hdr_cells[1].text = tgt_title

        for src_t, tgt_t in glossary_items:
            row_cells = table.add_row().cells
            row_cells[0].text = src_t
            row_cells[1].text = tgt_t
    else:
        doc_gloss.add_paragraph("No glossary terms could be extracted.")

    file_gloss = "3_Glossary_Terms.docx"
    doc_gloss.save(file_gloss)

    print("Step 4/4: Bundling files into a ZIP archive and triggering download...")
    with zipfile.ZipFile(ZIP_FILENAME, 'w') as zipf:
        zipf.write(file_orig)
        zipf.write(file_trans)
        zipf.write(file_gloss)

    print(f"Success! Ready for download: {ZIP_FILENAME}")
    files.download(ZIP_FILENAME)

# --- EXECUTE ---
generate_package()

Installing required packages (python-docx, spacy, deep-translator)...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 122.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 79.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 86.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 118.3 MB/s eta 0:00:00
✔ Download and installation successful
Y

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>